In [1]:
import os
import pandas as pd
import numpy as np

# Helper functions

In [2]:
settings = {
    "baseALNS": {"folder": "../results/baseALNS/Objective", "parameters": "7200_8_11_0.1_1_0.005_1e-8_100_1e-4_2_0"},
    "(a)-ini": {"folder": "../results/exactMethods/a-ini/Objective", "parameters": "86400_8_ini_0"},
    "(b)-ini": {"folder": "../results/exactMethods/b-ini/Objective", "parameters": "86400_8_ini_0"},
    "(c)-ini": {"folder": "../results/exactMethods/c-ini/Objective", "parameters": "86400_8_ini_0"},
    "(a)": {"folder": "../results/exactMethods/a/Objective", "parameters": "86400_8_noIni_0"},
    "(b)": {"folder": "../results/exactMethods/b/Objective", "parameters": "86400_8_noIni_0"},
    "(c)": {"folder": "../results/exactMethods/c/Objective", "parameters": "86400_8_noIni_0"},
    "noHeuristicOperator": {"folder": "../results/impactOperators/removingHeuristicOperator/Objective", "parameters": "7200_8_01_0.1_1_0.005_1e-8_100_1e-4_2_0"},
    "noMILPOperator": {"folder": "../results/impactOperators/removingMILPOperator/Objective", "parameters": "7200_8_10_0.1_1_0.005_1e-8_100_1e-4_2_0"},
    "exponent0": {"folder": "../results/mechanismWeightUpdate/expo0/Objective", "parameters": "7200_8_11_0.1_1_0.005_1e-8_100_1e-4_0_0"},
    "exponent1": {"folder": "../results/mechanismWeightUpdate/expo1/Objective", "parameters": "7200_8_11_0.1_1_0.005_1e-8_100_1e-4_1_0"},
    "exponent1.5": {"folder": "../results/mechanismWeightUpdate/expo1.5/Objective", "parameters": "7200_8_11_0.1_1_0.005_1e-8_100_1e-4_1.5_0"},
    "exponent2.5": {"folder": "../results/mechanismWeightUpdate/expo2.5/Objective", "parameters": "7200_8_11_0.1_1_0.005_1e-8_100_1e-4_2.5_0"},
    "exponent3": {"folder": "../results/mechanismWeightUpdate/expo3/Objective", "parameters": "7200_8_11_0.1_1_0.005_1e-8_100_1e-4_3_0"},
}

instances = [
 "S1", "S2", "S3",
 "M1", "M2", "M3",
 "L1", "L2", "L3"
 ]

In [3]:
def get_values(instance, settings, name):
    setting = settings[name]
    for stat in ("values", "mean", "best", "CV"):
        setting[stat + "_" + instance] = []

    inputFiles = os.listdir(setting["folder"])
    for inputFile in inputFiles:
        splited = inputFile.split("_")
        instanceRead = splited[1]
        if instanceRead == instance:
            with open(os.path.join(setting["folder"], inputFile), 'r') as file:
                value = float(file.read().strip())
                setting["values_" + instance].append(value)

    setting["mean_" + instance] = np.mean(setting["values_" + instance])
    setting["best_" + instance] = np.min(setting["values_" + instance])

    if len(setting["values_" + instance]) > 1 and np.isfinite(setting["mean_" + instance]):
        setting["CV_" + instance] = np.std(setting["values_" + instance]) / setting["mean_" + instance]
    else:
        setting["CV_" + instance] = np.nan

    return setting

# Comparison to Gurobi and the exact solution methods in Lee et al. (2022) (Section 5.2, Table 2)

In [4]:
index = pd.Index(["ALNS^avg_2h", "Gap_ALNS^(a)-ini", "Gap_ALNS^(b)-ini", "Gap_ALNS^(c)-ini", "Gap_ALNS^(a)", "Gap_ALNS^(b)", "Gap_ALNS^(c)", "ALNS^CV_2h"])
df = pd.DataFrame(index = index, columns = instances)
for instance in instances:
    baseALNS = get_values(instance, settings, "baseALNS")
    df.loc["ALNS^avg_2h", instance] = int(round(baseALNS["mean_" + instance], 0))
    df.loc["ALNS^CV_2h", instance] = str(round(100 * baseALNS["CV_" + instance], 1)) + " %"

    for name in ["(a)-ini", "(b)-ini", "(c)-ini"]: # ini
        exactMethod = get_values(instance, settings, name)
        gap = ((baseALNS["mean_" + instance] - exactMethod["mean_" + instance]) / exactMethod["mean_" + instance]) * 100
        df.loc["Gap_ALNS^" + name, instance] = str(round(gap, 1)) + " %"

    for name in ["(a)", "(b)", "(c)"]: # no ini
        exactMethod = get_values(instance, settings, name)
        if exactMethod["best_" + instance] == np.inf:
            df.loc["Gap_ALNS^" + name, instance] = "/"
        else:
            gap = ((baseALNS["mean_" + instance] - exactMethod["best_" + instance]) / exactMethod["best_" + instance]) * 100
            df.loc["Gap_ALNS^" + name, instance] = str(round(gap, 1)) + " %"
            if exactMethod["CV_" + instance] > 0.0:
                print("Warning: exact method " + name + " has CV > 0 for instance " + instance)
                print(exactMethod["values_" + instance])
df

,S1,S2,S3,M1,M2,M3,L1,L2,L3
ALNS^avg_2h,2218,1900,2431,4577,3891,4939,9850,8461,10802
Gap_ALNS^(a)-ini,-2.3 %,-2.3 %,-1.6 %,-5.9 %,-6.0 %,-6.1 %,-18.3 %,-31.1 %,-26.2 %
Gap_ALNS^(b)-ini,-1.7 %,-2.5 %,-2.7 %,-18.3 %,-14.9 %,-12.5 %,-37.4 %,-38.8 %,-34.1 %
Gap_ALNS^(c)-ini,-2.3 %,-2.4 %,-3.5 %,-41.3 %,-43.9 %,-40.5 %,-36.8 %,-38.4 %,-36.0 %
Gap_ALNS^(a),-5.8 %,-6.5 %,-0.2 %,-9.6 %,/,-10.7 %,/,/,/
Gap_ALNS^(b),-6.6 %,-5.9 %,-1.5 %,/,/,/,/,/,/
Gap_ALNS^(c),-1.1 %,-4.9 %,-7.4 %,/,/,/,/,/,/
ALNS^CV_2h,0.6 %,0.6 %,0.7 %,0.7 %,0.8 %,0.9 %,0.6 %,0.9 %,1.0 %


# Impact of each destroy-operator (Section 5.5, Table 4)

In [5]:
index = pd.Index(["Gap_noHeuristicOperator", "Gap_noMILPOperator"])
df = pd.DataFrame(index = index, columns = instances)
for instance in instances:
    baseALNS = get_values(instance, settings, "baseALNS")
    for name in ["noHeuristicOperator", "noMILPOperator"]:
        other = get_values(instance, settings, name)
        gap = ((other["mean_" + instance] - baseALNS["mean_" + instance]) / baseALNS["mean_" + instance]) * 100
        df.loc["Gap_" + name, instance] = str(round(gap, 1)) + " %"
df

,S1,S2,S3,M1,M2,M3,L1,L2,L3
Gap_noHeuristicOperator,0.8 %,0.1 %,0.1 %,1.5 %,0.3 %,1.3 %,2.0 %,1.6 %,1.4 %
Gap_noMILPOperator,5.9 %,6.4 %,4.9 %,7.5 %,7.4 %,8.1 %,3.2 %,3.1 %,3.7 %


# Impact of the mechanism updating the weights (Section 5.6, Table 5)

In [6]:
index = pd.Index(["Gap_exponent0", "Gap_exponent1", "Gap_exponent1.5", "Gap_exponent2.5", "Gap_exponent3"])
df = pd.DataFrame(index = index, columns = instances)
for instance in instances:
    baseALNS = get_values(instance, settings, "baseALNS")
    for name in ["exponent0", "exponent1", "exponent1.5", "exponent2.5", "exponent3"]:
        other = get_values(instance, settings, name)
        gap = ((other["mean_" + instance] - baseALNS["mean_" + instance]) / baseALNS["mean_" + instance]) * 100
        df.loc["Gap_" + name, instance] = str(round(gap, 1)) + " %"
df

,S1,S2,S3,M1,M2,M3,L1,L2,L3
Gap_exponent0,0.7 %,0.1 %,0.4 %,1.0 %,0.3 %,0.4 %,2.1 %,2.5 %,1.6 %
Gap_exponent1,0.1 %,-0.0 %,0.4 %,0.7 %,0.1 %,0.3 %,2.3 %,1.2 %,1.6 %
Gap_exponent1.5,0.3 %,0.1 %,0.2 %,0.4 %,-0.2 %,0.0 %,0.4 %,0.7 %,0.3 %
Gap_exponent2.5,0.7 %,1.0 %,0.5 %,1.6 %,1.4 %,1.0 %,0.3 %,0.3 %,0.6 %
Gap_exponent3,5.4 %,5.6 %,3.8 %,5.9 %,5.3 %,6.2 %,2.4 %,2.3 %,3.1 %
